### Demo: Data preparation for GPT-2 model fine-tuning

#### Step 1: Imports

In [1]:
from transformers import GPT2Tokenizer
from transformers import GPT2LMHeadModel
import torch
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import pandas as pd
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from datasets import DatasetDict, load_dataset
from typing import List, Dict, Tuple, Any

c:\Users\adeid\miniconda3\envs\slm_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Step 2: Configurations

In [2]:
MAX_LENGTH = 128
RANDOM_SEED = 100
DATA_LOADER_BATCH_SIZE = 32
TOKENIZATION_BATCH_SIZE = 256
TINY_STORIES_HF_DATASET_NAME = "roneneldan/TinyStories"
TINY_STORIES_HF_SPLIT_CONFIGS = {"train": "train[:5000]", "validation": "validation[:1000]"}
HUGGINGFACE_MODEL_NAME = "gpt2"
PARTITION_TYPES = {"train": "train", "validation": "validation", "test": "test"}
N_STORIES = 1000


#### Step 3: Prepare the dataset
- Get the **Tiny Stories** from Huggingface
- Tokenize the dataset
- Partition the dataset

In [3]:
class TinyStoriesDataPreprocessor:
    def __init__(
        self, 
        dataset_name=TINY_STORIES_HF_DATASET_NAME, 
        n_stories_split_configs: Dict[str, str] | None=TINY_STORIES_HF_SPLIT_CONFIGS):
        self.tokenizer = GPT2Tokenizer.from_pretrained(HUGGINGFACE_MODEL_NAME)
        # GPT-2 has no dedicated padding token; EOS is the standard safe substitute.
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.n_stories = n_stories_split_configs
        if n_stories_split_configs is not None:
            self.raw_dataset = DatasetDict({
                split_name: load_dataset(dataset_name, split=split_config)
                for split_name, split_config in n_stories_split_configs.items()
            })
        else:
            self.raw_dataset = load_dataset(dataset_name)
        self.train_input_ids = None
        self.validation_input_ids = None
        self.test_input_ids = None
        self.processed_dataset = None

    def split_story_by_sentence(self, example):
        """Split a story into its first-sentence prompt and remaining target."""
        text = example['text']
        parts = text.split(". ", 1)
        if len(parts) == 2:
            prompt = parts[0] + "."
            target = parts[1]
        else:
            prompt = text
            target = ""
        return {"prompt": prompt, "target": target}
    
    def tokenize_text(self, partition_type: str, batch_size: int=TOKENIZATION_BATCH_SIZE):
        dataset = self.processed_dataset[partition_type]
        tokenized_batches = []

        for start in tqdm(
            range(0, len(dataset), batch_size),
            desc=f"Tokenizing {partition_type}",
            unit="batch"
        ):
            rows = dataset[start:start + batch_size]
            texts = [
                f"{prompt}{self.tokenizer.eos_token}{target}{self.tokenizer.eos_token}"
                for prompt, target in zip(rows["prompt"], rows["target"])
            ]
            batch = self.tokenizer(
                texts,
                padding="max_length",
                truncation=True,
                max_length=MAX_LENGTH,
                return_tensors="pt"
            )
            tokenized_batches.append(batch["input_ids"])

        if not tokenized_batches:
            raise ValueError(f"Dataset partition '{partition_type}' is empty.")
        return torch.cat(tokenized_batches, dim=0)
    
    def preprocess_data(self, partition_types=PARTITION_TYPES):
        self.processed_dataset = self.raw_dataset.map(self.split_story_by_sentence)
        self.train_input_ids = self.tokenize_text(partition_types["train"])
        self.validation_input_ids = self.tokenize_text(partition_types["validation"])
        return {
            "train": self.train_input_ids,
            "validation": self.validation_input_ids,
        }

#### Step 4: Create data loaders
- Create train/validate data loaders to present the dataset to the GPT-2 model

In [4]:
class CustomDataset(Dataset):
  def __init__(self, input_ids):
    self.input_ids = input_ids

  def __len__(self):
    return len(self.input_ids)

  def __getitem__(self, idx):
    return {'input_ids': self.input_ids[idx]}

def create_dataloaders(
    train_input_ids: torch.Tensor, 
    validation_input_ids: torch.Tensor, 
    batch_size: int=DATA_LOADER_BATCH_SIZE
) -> Tuple[DataLoader, DataLoader]:
    train_dataset = CustomDataset(train_input_ids)
    validation_dataset = CustomDataset(validation_input_ids)

    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    validation_dataloader = DataLoader(validation_dataset, batch_size=batch_size)

    return train_dataloader, validation_dataloader

#### Step 5: Create GPT-2 classification model

In [5]:
preprocessor = TinyStoriesDataPreprocessor()
dataset = preprocessor.preprocess_data()
train_loader, val_loader = create_dataloaders(
    dataset["train"], dataset["validation"])

model = GPT2LMHeadModel.from_pretrained(HUGGINGFACE_MODEL_NAME)

Tokenizing validation: 100%|██████████| 4/4 [00:01<00:00,  3.06batch/s]
